In [0]:
%run "./0.0 Utils"

In [0]:
# =========================================================
# Widget definitions — visible at the top of the notebook
# =========================================================

# Remove any existing widgets to avoid duplicates when re-running
dbutils.widgets.removeAll()

# Create text widgets for Unity Catalog parameters
dbutils.widgets.text("CATALOG_NAME", "workspace", "UC Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "default", "UC Schema Name")
dbutils.widgets.text("TABLE_NAME", "fct_activity", "UC Table Name")
dbutils.widgets.text("CHECKPOINT_SUBPATH", "rt/checkpoints/fct_activity_viewer", "Checkpoint Subpath")

# =========================================================
# Retrieve widget values
# =========================================================
CATALOG_NAME       = dbutils.widgets.get("CATALOG_NAME").strip()
SCHEMA_NAME        = dbutils.widgets.get("SCHEMA_NAME").strip()
TABLE_NAME         = dbutils.widgets.get("TABLE_NAME").strip()
CHECKPOINT_SUBPATH = dbutils.widgets.get("CHECKPOINT_SUBPATH").strip().strip("/")

# Preview values
print(f"UC Catalog: {CATALOG_NAME}")
print(f"UC Schema: {SCHEMA_NAME}")
print(f"UC Table: {TABLE_NAME}")
print(f"Checkpoint: {CHECKPOINT_SUBPATH}")



In [0]:
from pyspark.sql import functions as F, types as T

# =========================================================
# Unity Catalog table resolution
# =========================================================

# Build fully qualified table name
full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# Use Unity Catalog Volume for checkpoint (required for Serverless)
# Ensure the checkpoints volume exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.checkpoints")
checkpoint_base = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/checkpoints/{CHECKPOINT_SUBPATH}"

print(f"Reading streaming data from Unity Catalog table: {full_table_name}")
print(f"Checkpoint location (UC Volume): {checkpoint_base}")

# =========================================================
# Streaming read from Unity Catalog Delta table
# =========================================================

src = (
    spark.readStream
         .format("delta")
         .table(full_table_name)
)

# =========================================================
# Basic visualization: minute-level counts by activity_type
# =========================================================

agg_1min = (
    src
        .groupBy(F.window("process_run_ts", "1 minute"), F.col("activity_type"))
        .count()
        .select(
            F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "activity_type",
            "count"
        )
        .orderBy("window_start")
)

# Show the aggregate as a table; in the Databricks UI, switch to a chart.
# Note: checkpointLocation is required for Serverless compute
checkpoint_path = f"{checkpoint_base}/agg_1min"
print(f"Using checkpoint: {checkpoint_path}")
display(agg_1min, streamName="fct_activity_agg_1min", checkpointLocation=checkpoint_path)
